# AI Tutor — OSS **Qwen + Gemma** eval on Google Colab (free T4)  ·  *Improved Evaluation (results3)*

Evaluate the open-source **Qwen** and **Gemma** tutor models that an 8 GB laptop
can't run, using the **same harness** (tutor = OSS via Ollama; judge + student-sim
= Anthropic).

**Models (Cell 8).** The free-T4 tier runs `qwen2.5:{0.5,1.5,3,7,14}b` and
`gemma3:{1,4,12}b`. The **XL tier is now active** (no longer commented):
`qwen2.5:32b/72b`, `gemma3:27b`, plus `qwen3:30b-a3b`, `qwen3.6:27b`,
`qwen3.6:35b-a3b` — these need an **A100 (80 GB for the 72b) / Colab-Pro**
runtime and will OOM a free T4. Each model is auto-tuned by
`apps/llm/model_profiles.py`:
- **Qwen2.5** → temp **0.7 / top_p 0.8 / top_k 20**, **Markdown** Block-0
  (targeted rules + few-shot).
- **Gemma 3** → temp **1.0 / top_p 0.95 / top_k 64** (Gemma's documented default),
  **XML** Block-0 + targeted rules (Google lineage favours XML here).

Results land in **`offline_eval/results3/`** so they join the Gemini + Qwen-MaaS
cloud re-run on the new board.

> **Gemma tool-calling caveat.** The engine requires tool calls; Gemma's
> tool-calling via Ollama is weaker than Qwen's, so it leans on the engine's
> text-tool-call recovery + the "never emit tool syntax" rule. If Cell 9 shows
> Gemma erroring every scenario with *"does not support tools"*, that Ollama build
> can't run it through the tool-required harness — check the per-model `.log`.

> Requires the `pixeldesignlabs-dev-portuguese` branch to contain the bottleneck-fix commit (B1/B2 engine
> fixes, per-family prompts incl. Gemma routing, rubric n/a, dataset reference
> fixes). Cell 2 clones that branch, so make sure it's pushed before running.

**Before you start**
1. Runtime → **Change runtime type → T4 GPU**.
2. Add these **Colab Secrets** (🔑 icon in the left sidebar), each toggled
   *Notebook access ON*:
   - `GH_TOKEN` — a GitHub **classic** Personal Access Token with the **`repo`**
     scope. Make it at github.com/settings/tokens → *Generate new token
     (classic)* → check **repo**. A classic token works on `eai6/ai-tutor` because you
     are a **collaborator** (a fine-grained token would only work if you *owned*
     the repo).
   - `ANTHROPIC_API_KEY` — required (judge + student-simulator).
   - `GOOGLE_API_KEY` and `OPENAI_API_KEY` — keep these too so the judge/grader
     cross-vendor cascade matches the laptop runs (comparable scores).

**T4 fits models up to ~14B q4.** For the bigger A100/Colab-Pro tier (commented
out in Cell 8), use a Colab Pro A100 runtime — nothing else changes.

## Cell 1 — confirm GPU + mount Drive (Drive persists results across disconnects)

In [1]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-0f925da3-d43f-69e9-8724-6f200daea768)
Mounted at /content/drive


## Cell 2 — clone the repo (branch `pixeldesignlabs-dev-portuguese`) using the GH_TOKEN classic PAT

In [2]:
from google.colab import userdata
import subprocess, os
tok = (userdata.get('GH_TOKEN') or '').strip()   # strip stray spaces/newlines
assert tok and ' ' not in tok, "GH_TOKEN missing or contains a space — re-save the secret with no whitespace"
url = f"https://{tok}@github.com/eai6/ai-tutor.git"
subprocess.run(['rm', '-rf', '/content/ai-tutor'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'pixeldesignlabs-dev-portuguese', url, '/content/ai-tutor'], check=True)
os.chdir('/content/ai-tutor')
print('cloned at', os.getcwd())

cloned at /content/ai-tutor


## Cell 3 — fix hardcoded laptop paths (essential)

In [3]:
!sed -i 's#/home/daniel/Documents/work/Nyansapo/web/ai-tutor#/content/ai-tutor#g; s#\$ROOT/venv/bin/python#python#g; s#venv/bin/python#python#g' offline_eval/*.py offline_eval/*.sh

## Cell 4 — install deps + start Ollama (a few min; ignore pip resolver warnings)

In [4]:
!pip install -q -r requirements.txt
# The Ollama installer is now zstd-compressed; the Colab VM lacks zstd, so install
# it first (otherwise the installer aborts and `ollama` is never created).
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, shutil
assert shutil.which('ollama'), "ollama did not install — check the install output above (zstd?)"
subprocess.Popen(['ollama', 'serve'],
                 stdout=open('/content/ollama.log', 'w'),
                 stderr=subprocess.STDOUT)
for _ in range(30):
    if subprocess.run(['bash', '-c', 'ollama list'], capture_output=True).returncode == 0:
        print('ollama ready'); break
    time.sleep(2)
else:
    print('ollama NOT ready — check /content/ollama.log')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 10.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.9/405.9 kB 33.4 MB/s eta 0:00

## Cell 5 — **required** — write .env from Colab Secrets
`.env` isn't in the repo (gitignored). Keep **all three** keys so the judge/grader cascade matches the laptop runs (comparable scores).

In [5]:
from google.colab import userdata
open('.env', 'w').write(
    "SECRET_KEY=colab-eval\nDEBUG=True\nEMBEDDING_BACKEND=sqlite\n"
    f"ANTHROPIC_API_KEY={userdata.get('ANTHROPIC_API_KEY')}\n"
    f"GOOGLE_API_KEY={userdata.get('GOOGLE_API_KEY')}\n"
    f"OPENAI_API_KEY={userdata.get('OPENAI_API_KEY')}\n")
print('.env written')

.env written


## Cell 6 — fresh DB + eval fixtures

In [6]:
!python manage.py migrate
!python manage.py loaddata evals/fixtures/institution.json evals/fixtures/lessons.json

Operations to perform:
  Apply all migrations: accounts, admin, auth, benchmark, contenttypes, curriculum, dashboard, llm, media_library, safety, sessions, support, token_blacklist, tutoring
Running migrations:
  Applying contenttypes.0001_initial... OK
  Applying auth.0001_initial... OK
  Applying accounts.0001_initial... OK
  Applying llm.0001_initial... OK
  Applying llm.0002_promptpack_content_generation_prompt_and_more... OK
  Applying accounts.0002_studentprofile... OK
  Applying accounts.0003_staffinvitation... OK
  Applying accounts.0004_alter_membership_role_alter_staffinvitation_role... OK
  Applying accounts.0005_institution_accent_color_institution_custom_css_and_more... OK
  Applying accounts.0006_platformconfig_remove_institution_custom_css... OK
  Applying accounts.0007_alter_membership_role_alter_staffinvitation_email_and_more... OK
  Applying accounts.0008_move_branding_to_platformconfig... OK
  Applying llm.0003_make_promptpack_institution_nullable... OK
  Applying ac

## Cell 7 — persist results to Drive (seed with the committed cloud results3, then symlink)
Writes into **`results3/`** so the OSS Qwen models join the Gemini + Qwen-MaaS cloud re-run. Seeds from any committed cloud `results3/*.json` so the combined leaderboard shows cloud + OSS together, and survives Colab disconnects.

In [7]:
!mkdir -p /content/drive/MyDrive/ai-tutor-eval-results3
# seed the Drive folder with the cloud results committed in the repo (no-clobber)
!cp -n offline_eval/results3/*.json /content/drive/MyDrive/ai-tutor-eval-results3/ 2>/dev/null || true
!rm -rf offline_eval/results3 && ln -s /content/drive/MyDrive/ai-tutor-eval-results3 offline_eval/results3
!ls offline_eval/results3/

gemini-2.5-flash.json  gemma3_4b.json	  qwen2.5_3b.json
gemini-2.5-pro.json    gemma3_4b.log	  qwen2.5_3b.log
gemini-3.1-pro.json    qwen2.5_0.5b.json  qwen2.5_7b.json
gemini-3.5-flash.json  qwen2.5_0.5b.log   qwen2.5_7b.log
gemma3_12b.json        qwen2.5_14b.json   qwen3-next-80b-instruct.json
gemma3_12b.log	       qwen2.5_14b.log	  qwen3-next-80b-thinking.json
gemma3_1b.json	       qwen2.5_1.5b.json
gemma3_1b.log	       qwen2.5_1.5b.log


## Cell 8 — OSS Qwen + Gemma matrix + seed configs
The **free-T4 tier** (≤~14B q4) holds the full small-to-mid Qwen2.5 range plus Gemma 3. The **XL tier is active** (`qwen2.5:32b/72b`, `gemma3:27b`, and the Qwen3 / Qwen3.6 MoE+dense models) — these need an **A100 (80 GB for the 72b) / Colab-Pro** runtime and OOM a free T4. ~20–40 min each (XL longer); it's **resume-safe** across Colab disconnects (done models are skipped). Trim the list to fit your runtime + session budget.

In [8]:
open('offline_eval/models.txt', 'w').write('''\
# ============ T4 (free Colab, 16GB) tier — fits ~14B q4 ============
# Qwen2.5 — full small-to-mid range
qwen2.5:0.5b         big
qwen2.5:1.5b         big
qwen2.5:3b           big
qwen2.5:7b           big
qwen2.5:14b          big
# Gemma 3 — Google OSS (weaker Ollama tool-calling; see caveat at top)
gemma3:1b            big
gemma3:4b            big
gemma3:12b           big

# ============ XL tier — >16GB VRAM; A100 (80GB for 72b) / Colab-Pro; OOMs a T4 ===
qwen2.5:32b          xl
qwen2.5:72b          xl     # q4 ~47GB — needs the 80GB A100
gemma3:27b           xl
qwen3:30b-a3b        xl     # Qwen3 30B MoE (3B active)
qwen3.6:27b          xl     # Qwen3.6 dense 27B
qwen3.6:35b-a3b      xl     # Qwen3.6 35B MoE (3B active)
''')
!python offline_eval/seed_ollama_configs.py
# Show the per-family sampling each model will use (from apps/llm/model_profiles).
# This is the "improved" tuning — confirm it resolves before spending GPU time.
import django, os
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')
django.setup()
from apps.llm.model_profiles import get_model_profile
print(f"{'MODEL':<22} {'FAMILY':<9} {'MODE':<11} {'MAXTOK':>7}  SAMPLING")
print('-' * 78)
for line in open('offline_eval/models.txt'):
    tag = line.split('#')[0].split()[0] if line.split('#')[0].split() else ''
    if not tag:
        continue
    p = get_model_profile(f'local_ollama/{tag}')
    if p:
        print(f"{tag:<22} {p.family:<9} {p.mode:<11} {p.max_tokens:>7}  {p.sampling_dict()}")
    else:
        print(f"{tag:<22} (no profile — runs at engine default)")

  created: local_ollama/qwen2.5:0.5b
  created: local_ollama/qwen2.5:1.5b
  created: local_ollama/qwen2.5:3b
  created: local_ollama/qwen2.5:7b
  created: local_ollama/qwen2.5:14b
  created: local_ollama/gemma3:1b
  created: local_ollama/gemma3:4b
  created: local_ollama/gemma3:12b
  created: local_ollama/qwen2.5:32b
  created: local_ollama/qwen2.5:72b
  created: local_ollama/gemma3:27b
  created: local_ollama/qwen3:30b-a3b
  created: local_ollama/qwen3.6:27b
  created: local_ollama/qwen3.6:35b-a3b

Seeded 14 ollama configs (14 created, 0 updated).
MODEL                  FAMILY    MODE         MAXTOK  SAMPLING
------------------------------------------------------------------------------
qwen2.5:0.5b           qwen      instruct       1024  {'temperature': 0.7, 'top_p': 0.8, 'top_k': 20}
qwen2.5:1.5b           qwen      instruct       1024  {'temperature': 0.7, 'top_p': 0.8, 'top_k': 20}
qwen2.5:3b             qwen      instruct       1024  {'temperature': 0.7, 'top_p': 0.8, 'top_k': 2

## Cell 8b — reclaim disk BEFORE the sweep (important on a resumed session)
A reconnected / re-cloned Colab can start with old pulled model weights + caches still on disk (the last sweep left it ~half full). This clears every previously-pulled Ollama model and the package caches **before** the first download, then prints free space. On a fresh VM it's a harmless no-op.

In [9]:
import subprocess
def _df(tag):
    print(f"--- disk {tag} ---\n" + subprocess.run(['df','-h','/'],capture_output=True,text=True).stdout)
_df('BEFORE cleanup')
# remove every model the Ollama server currently holds (server-mediated → frees blobs)
!ollama list 2>/dev/null | tail -n +2 | awk '{print $1}' | xargs -r -n1 ollama rm 2>/dev/null || true
# backstop: wipe any stray model store + pip/apt/HF caches (server re-creates on next pull)
!rm -rf /root/.ollama/models/blobs/* /root/.ollama/models/manifests/* offline_eval/ollama_models/* 2>/dev/null || true
!pip cache purge 2>/dev/null || true
!apt-get clean 2>/dev/null || true
!rm -rf /root/.cache/huggingface /root/.cache/pip 2>/dev/null || true
_df('AFTER cleanup')

--- disk BEFORE cleanup ---
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   56G  180G  24% /

Files removed: 839
--- disk AFTER cleanup ---
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   54G  183G  23% /



## Cell 9 — run the sweep (pulls + scores each model; resume-safe; ~20–40 min/model on T4)
`RESULTS_DIR=…/results3` puts these on the new board (Gemini + Qwen-MaaS + OSS). `CLEANUP_MODELS=1` deletes each model's weights from disk **right after** it's scored, so peak disk ≈ one model at a time and it never fills up (results are already on Drive, so a re-run still skips done models).

In [ ]:
!RESULTS_DIR=$PWD/offline_eval/results3 SIMPLE_TUTOR_ENGINE=1 CLEANUP_MODELS=1 bash offline_eval/run_matrix.sh

>> Seeding local_ollama ModelConfig rows...
  updated: local_ollama/qwen2.5:0.5b
  updated: local_ollama/qwen2.5:1.5b
  updated: local_ollama/qwen2.5:3b
  updated: local_ollama/qwen2.5:7b
  updated: local_ollama/qwen2.5:14b
  updated: local_ollama/gemma3:1b
  updated: local_ollama/gemma3:4b
  updated: local_ollama/gemma3:12b
  updated: local_ollama/qwen2.5:32b
  updated: local_ollama/qwen2.5:72b
  updated: local_ollama/gemma3:27b
  updated: local_ollama/qwen3:30b-a3b
  updated: local_ollama/qwen3.6:27b
  updated: local_ollama/qwen3.6:35b-a3b

Seeded 14 ollama configs (0 created, 14 updated).
>> Engine: SIMPLE_TUTOR_ENGINE=1   Mode: --single-turn
>> Model weights: /content/ai-tutor/offline_eval/ollama_models

==================== qwen2.5:0.5b (big) — already done, skipping ====================

==================== qwen2.5:1.5b (big) — already done, skipping ====================

==================== qwen2.5:3b (big) — already done, skipping ====================

==================== qw

## Cell 9b — reclaim disk AFTER the sweep (final backstop)
`CLEANUP_MODELS=1` already removes each model as it finishes; this drops anything left and prints free space. Results are safe on Drive (Cell 7).

In [ ]:
!ollama list 2>/dev/null | tail -n +2 | awk '{print $1}' | xargs -r -n1 ollama rm 2>/dev/null || true
!rm -rf /root/.ollama/models/blobs/* /root/.ollama/models/manifests/* offline_eval/ollama_models/* 2>/dev/null || true
!pip cache purge 2>/dev/null || true
!apt-get clean 2>/dev/null || true
import subprocess
print(subprocess.run(['df','-h','/'],capture_output=True,text=True).stdout)

## Cell 10 — combined results3 leaderboard (cloud + OSS Qwen/Gemma; run anytime)

In [ ]:
!RESULTS_DIR=$PWD/offline_eval/results3 python offline_eval/aggregate.py

## After a Colab disconnect (free tier: ~90 min idle / ~12 h max)
Re-run **Cells 1–8**, then **Cell 9** again. Because results live on Drive (Cell 7),
`run_matrix.sh` **skips already-scored models** and continues.

To pick up new commits on the branch, just re-run **Cell 2** (it re-clones).

**Tips:** keep the tab active (free Colab kills idle sessions); a model interrupted
mid-run restarts (resume only skips *completed* models); each model is also bound on
the Anthropic judge calls, so plan 1–3 models per session.

To pull these results back to your laptop: copy the JSONs from
`MyDrive/ai-tutor-eval-results3/` into the repo's `offline_eval/results3/` and run
`RESULTS_DIR=offline_eval/results3 python offline_eval/aggregate.py`.

**If a Qwen model leaks tool calls as text** (the engine's text-tool-call recovery
should now catch `record_answer(...)`): set `OLLAMA_DEBUG_RAW=1` before Cell 9, then
check `offline_eval/results3/qwen2.5_14b.log` for an `[OllamaToolLeak]` line and
share it — the parser can then be extended to match that exact format.